# Module 3: Production Deployment with AgentCore

## Overview

In this module, you'll deploy the multi-agent customer service system to production using:

- **AgentCore Runtime**: Managed hosting for the agent application
- **AgentCore Gateway**: MCP tool integration (optional)
- **AgentCore Memory**: Conversation persistence
- **AgentCore Observability**: CloudWatch integration for tracing

### Learning Objectives
1. Deploy agents to AgentCore Runtime
2. Configure observability with CloudWatch
3. Set up session management with AgentCore Memory
4. Test the production deployment

### Time: ~60 minutes

## Step 1: Environment Setup

In [1]:
# Install dependencies
!pip install -q bedrock-agentcore bedrock-agentcore-starter-toolkit boto3


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: /opt/homebrew/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip


In [2]:
import boto3
import json
import os
import time
from datetime import datetime

# Get AWS configuration
session = boto3.Session()
REGION = session.region_name or 'us-east-1'
ACCOUNT_ID = boto3.client('sts').get_caller_identity()['Account']

print(f"AWS Region: {REGION}")
print(f"Account ID: {ACCOUNT_ID}")

AWS Region: us-west-2
Account ID: 537124949553


In [3]:
# Load baseline metrics from Module 2
try:
    %store -r baseline_metrics
    print("Loaded baseline metrics from Module 2")
    print(f"Goal Success Rate: {baseline_metrics.get('goal_success_rate', 'N/A'):.2%}")
except:
    print("Could not load baseline metrics - will establish new baseline in production")
    baseline_metrics = {}

Loaded baseline metrics from Module 2
Goal Success Rate: 65.00%


## Step 2: Create Requirements File

In [4]:
%%writefile requirements.txt
bedrock-agentcore
strands-agents
boto3
aws-opentelemetry-distro

Overwriting requirements.txt


## Step 3: Review the Production Application

The `app.py` file contains our production-ready multi-agent system with:
- Inline tool definitions for single-file deployment
- Lazy agent initialization for cold start optimization
- Proper logging for observability
- AgentCore entrypoint decorator
- Global cross-region inference with Claude Sonnet 4.5 and Haiku 4.5

In [5]:
# Review the app structure
with open('app.py', 'r') as f:
    content = f.read()

# Show key sections
print("Production Application Structure:")
print("="*50)
print("\n1. Tool Definitions (Order, Product, Account)")
print("2. Agent Creation Functions (Haiku 4.5 for sub-agents, Sonnet 4.5 for orchestrator)")
print("3. AgentCore Entrypoint with session handling")
print(f"\nTotal lines: {len(content.splitlines())}")

Production Application Structure:

1. Tool Definitions (Order, Product, Account)
2. Agent Creation Functions (Haiku 4.5 for sub-agents, Sonnet 4.5 for orchestrator)
3. AgentCore Entrypoint with session handling

Total lines: 367


## Step 4: Create Execution Role

The execution role needs permissions for:
- Amazon Bedrock model invocation
- DynamoDB access for orders and accounts
- Bedrock Knowledge Base retrieval
- CloudWatch logging and X-Ray tracing

In [11]:
import json

iam_client = boto3.client('iam')
AGENT_NAME = 'ecommerce_customer_service'  # ✅ Use underscores, not hyphens
ROLE_NAME = f'AgentCore-{AGENT_NAME}-role'

# Trust policy for AgentCore
trust_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "bedrock-agentcore.amazonaws.com"},
        "Action": "sts:AssumeRole"
    }]
}

# Permissions policy
permissions_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "BedrockAccess",
            "Effect": "Allow",
            "Action": [
                "bedrock:InvokeModel",
                "bedrock:InvokeModelWithResponseStream"
            ],
            "Resource": "*"
        },
        {
            "Sid": "BedrockKBAccess",
            "Effect": "Allow",
            "Action": [
                "bedrock:Retrieve",
                "bedrock:RetrieveAndGenerate"
            ],
            "Resource": "*"
        },
        {
            "Sid": "ECRAccess",
            "Effect": "Allow",
            "Action": [
                "ecr:GetAuthorizationToken",
                "ecr:BatchCheckLayerAvailability",
                "ecr:GetDownloadUrlForLayer",
                "ecr:BatchGetImage"
            ],
            "Resource": "*"
        },
        {
            "Sid": "DynamoDBAccess",
            "Effect": "Allow",
            "Action": [
                "dynamodb:GetItem",
                "dynamodb:PutItem",
                "dynamodb:UpdateItem",
                "dynamodb:Query",
                "dynamodb:Scan"
            ],
            "Resource": [
                f"arn:aws:dynamodb:{REGION}:{ACCOUNT_ID}:table/ecommerce-workshop-*"
            ]
        },
        {
            "Sid": "SSMAccess",
            "Effect": "Allow",
            "Action": ["ssm:GetParameter"],
            "Resource": f"arn:aws:ssm:{REGION}:{ACCOUNT_ID}:parameter/ecommerce-workshop-*"
        },
        {
            "Sid": "CloudWatchLogs",
            "Effect": "Allow",
            "Action": [
                "logs:CreateLogGroup",
                "logs:CreateLogStream",
                "logs:PutLogEvents"
            ],
            "Resource": "*"
        },
        {
            "Sid": "XRayTracing",
            "Effect": "Allow",
            "Action": [
                "xray:PutTraceSegments",
                "xray:PutTelemetryRecords"
            ],
            "Resource": "*"
        }
    ]
}

# Create role
try:
    response = iam_client.create_role(
        RoleName=ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description='Execution role for E-Commerce Customer Service Agent'
    )
    ROLE_ARN = response['Role']['Arn']
    print(f"Created role: {ROLE_ARN}")
    
    # Attach permissions
    iam_client.put_role_policy(
        RoleName=ROLE_NAME,
        PolicyName=f'{ROLE_NAME}-policy',
        PolicyDocument=json.dumps(permissions_policy)
    )
    print("Attached permissions policy")
    
    # Wait for role propagation
    print("Waiting for IAM role propagation...")
    time.sleep(10)
    
except iam_client.exceptions.EntityAlreadyExistsException:
    ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/{ROLE_NAME}"
    print(f"Role already exists: {ROLE_ARN}")
    
    # Update the policy for existing role
    try:
        iam_client.put_role_policy(
            RoleName=ROLE_NAME,
            PolicyName=f'{ROLE_NAME}-policy',
            PolicyDocument=json.dumps(permissions_policy)
        )
        print("Updated permissions policy with ECR access")
    except Exception as e:
        print(f"Warning: Could not update policy: {e}")

Role already exists: arn:aws:iam::537124949553:role/AgentCore-ecommerce_customer_service-role
Updated permissions policy with ECR access


## Step 5: Deploy to AgentCore Runtime

In [12]:
from bedrock_agentcore_starter_toolkit import Runtime

# Initialize Runtime
runtime = Runtime()

# Configure deployment
config = runtime.configure(
    entrypoint="app.py",
    execution_role=ROLE_ARN,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=REGION,
    agent_name=AGENT_NAME
)

print("Configuration complete:")
print(f"  Agent Name: {AGENT_NAME}")
print(f"  Region: {REGION}")
print(f"  Execution Role: {ROLE_ARN}")

Entrypoint parsed: file=/Users/mmelli/Library/CloudStorage/OneDrive-amazon.com/GenAI-SSA/github/ecommerce-agent-workshop/03-production-deployment/app.py, bedrock_agentcore_name=app
Memory disabled - agent will be stateless
Configuring BedrockAgentCore agent: ecommerce_customer_service
Memory disabled
Network mode: PUBLIC
Generated Dockerfile: Dockerfile
Generated .dockerignore: /Users/mmelli/Library/CloudStorage/OneDrive-amazon.com/GenAI-SSA/github/ecommerce-agent-workshop/03-production-deployment/.dockerignore
Keeping 'ecommerce_customer_service' as default agent
Bedrock AgentCore configured: /Users/mmelli/Library/CloudStorage/OneDrive-amazon.com/GenAI-SSA/github/ecommerce-agent-workshop/03-production-deployment/.bedrock_agentcore.yaml


Configuration complete:
  Agent Name: ecommerce_customer_service
  Region: us-west-2
  Execution Role: arn:aws:iam::537124949553:role/AgentCore-ecommerce_customer_service-role


In [13]:
# Launch deployment
print("Starting deployment to AgentCore Runtime...")
print("This will:")
print("  1. Build container image")
print("  2. Push to ECR")
print("  3. Create AgentCore Runtime")
print("  4. Configure observability")
print("\nThis may take 5-10 minutes...\n")

launch_result = runtime.launch()
print(f"\nLaunch initiated!")

🚀 Launching Bedrock AgentCore (cloud mode - RECOMMENDED)...
   • Deploy Python code directly to runtime
   • No Docker required (DEFAULT behavior)
   • Production-ready deployment

💡 Deployment options:
   • runtime.launch()                → Cloud (current)
   • runtime.launch(local=True)      → Local development
Memory disabled - skipping memory creation
Starting CodeBuild ARM64 deployment for agent 'ecommerce_customer_service' to account 537124949553 (us-west-2)
Setting up AWS resources (ECR repository, execution roles)...
Getting or creating ECR repository for agent: ecommerce_customer_service


Starting deployment to AgentCore Runtime...
This will:
  1. Build container image
  2. Push to ECR
  3. Create AgentCore Runtime
  4. Configure observability

This may take 5-10 minutes...



ECR repository available: 537124949553.dkr.ecr.us-west-2.amazonaws.com/bedrock-agentcore-ecommerce_customer_service
Using execution role from config: arn:aws:iam::537124949553:role/AgentCore-ecommerce_customer_service-role
Preparing CodeBuild project and uploading source...


✅ Reusing existing ECR repository: 537124949553.dkr.ecr.us-west-2.amazonaws.com/bedrock-agentcore-ecommerce_customer_service


Getting or creating CodeBuild execution role for agent: ecommerce_customer_service
Role name: AmazonBedrockAgentCoreSDKCodeBuild-us-west-2-7874cf28f3
Reusing existing CodeBuild execution role: arn:aws:iam::537124949553:role/AmazonBedrockAgentCoreSDKCodeBuild-us-west-2-7874cf28f3
Using dockerignore.template with 46 patterns for zip filtering
Uploaded source to S3: ecommerce_customer_service/source.zip
Updated CodeBuild project: bedrock-agentcore-ecommerce_customer_service-builder
Starting CodeBuild build (this may take several minutes)...
Starting CodeBuild monitoring...
🔄 QUEUED started (total: 0s)
✅ QUEUED completed in 1.2s
🔄 PROVISIONING started (total: 1s)
✅ PROVISIONING completed in 7.2s
🔄 DOWNLOAD_SOURCE started (total: 9s)
✅ DOWNLOAD_SOURCE completed in 1.2s
🔄 BUILD started (total: 10s)
✅ BUILD completed in 10.8s
🔄 POST_BUILD started (total: 21s)
✅ POST_BUILD completed in 6.0s
🔄 FINALIZING started (total: 27s)
✅ FINALIZING completed in 1.2s
🔄 COMPLETED started (total: 28s)
✅ COMP


Launch initiated!


In [14]:
# Wait for deployment to complete
print("Waiting for deployment to complete...")

terminal_states = {'READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED'}

while True:
    status_result = runtime.status()
    status = status_result.endpoint.get('status', 'UNKNOWN')
    
    print(f"  Status: {status}")
    
    if status in terminal_states:
        break
    
    time.sleep(15)

if status == 'READY':
    print("\n" + "="*50)
    print("DEPLOYMENT SUCCESSFUL!")
    print("="*50)
    print(f"Agent Runtime ID: {launch_result.agent_id}")
    print(f"Agent ARN: {launch_result.agent_arn}")
else:
    print(f"\nDeployment ended with status: {status}")

Waiting for deployment to complete...


Retrieved Bedrock AgentCore status for: ecommerce_customer_service


  Status: READY

DEPLOYMENT SUCCESSFUL!
Agent Runtime ID: ecommerce_customer_service-j10dRbGkJf
Agent ARN: arn:aws:bedrock-agentcore:us-west-2:537124949553:runtime/ecommerce_customer_service-j10dRbGkJf


## Step 6: Test Production Deployment

In [15]:
# Test the deployed agent
print("Testing Production Deployment")
print("="*50)

test_prompts = [
    "What's the status of order ORD-2024-10002?",
    "Do you have any wireless headphones under $100?",
    "What are the benefits of Gold membership?"
]

for i, prompt in enumerate(test_prompts, 1):
    print(f"\nTest {i}: {prompt}")
    print("-"*40)
    
    try:
        response = runtime.invoke({"prompt": prompt})
        print(f"Response: {response.get('response', response)[:500]}...")
    except Exception as e:
        print(f"Error: {e}")

Testing Production Deployment

Test 1: What's the status of order ORD-2024-10002?
----------------------------------------
Response: ['{"response": "Your order **ORD-2024-10002** has been **shipped**! 📦\\n\\nHere are the details:\\n\\n**Order Information:**\\n- **Order Date:** December 28, 2024\\n- **Status:** Shipped ✅\\n- **Total:** $359.97\\n\\n**Items Ordered:**\\n- Smart Watch Pro (1x) - $299.99\\n- Watch Band - Leather (2x) - $29.99 each\\n\\n**Shipping Details:**\\n- **Carrier:** UPS\\n- **Tracking Number:** 1Z999AA10123456784\\n\\nYour package is currently in transit. You can track it using the UPS tracking number provided. Is there anything else you\'d like to know about this order?\\n"}']...

Test 2: Do you have any wireless headphones under $100?
----------------------------------------
Response: ['{"response": "I apologize, but I\'m currently experiencing a technical issue accessing our product database to search for wireless headphones under $100. \\n\\nHowever, I can stil

## Step 7: Configure Observability

AgentCore automatically integrates with CloudWatch for observability. Let's verify the setup.

In [ ]:
# Check CloudWatch log group
logs_client = boto3.client('logs', region_name=REGION)

log_group_prefix = f'/aws/bedrock-agentcore/{AGENT_NAME}'

try:
    response = logs_client.describe_log_groups(logGroupNamePrefix=log_group_prefix)
    log_groups = response.get('logGroups', [])
    
    if log_groups:
        print("CloudWatch Log Groups:")
        for lg in log_groups:
            print(f"  - {lg['logGroupName']}")
    else:
        print(f"No log groups found with prefix: {log_group_prefix}")
        print("Log groups will be created on first invocation.")
except Exception as e:
    print(f"Error checking logs: {e}")

In [ ]:
# View recent traces (if available)
print("\nObservability Dashboard Links:")
print("="*50)
print(f"\nCloudWatch Console:")
print(f"  https://{REGION}.console.aws.amazon.com/cloudwatch/home?region={REGION}#gen-ai-observability/agent-core/agents")
print(f"\nX-Ray Traces:")
print(f"  https://{REGION}.console.aws.amazon.com/xray/home?region={REGION}#/traces")

## Step 8: Session Management with AgentCore Memory (Optional)

For multi-turn conversations, AgentCore provides built-in session management.

In [ ]:
# Test multi-turn conversation with session
import uuid

session_id = str(uuid.uuid4())
print(f"Starting conversation session: {session_id}")
print("="*50)

conversation = [
    "Hi, I'd like to check on my order",
    "The order number is ORD-2024-10002",
    "When will it arrive?"
]

for i, message in enumerate(conversation, 1):
    print(f"\nTurn {i}")
    print(f"Customer: {message}")
    
    try:
        response = runtime.invoke(
            {"prompt": message},
            session_id=session_id
        )
        print(f"Agent: {response.get('response', response)[:300]}...")
    except Exception as e:
        print(f"Error: {e}")

## Step 9: Save Deployment Information

In [ ]:
# Save deployment info for Module 4
deployment_info = {
    'agent_name': AGENT_NAME,
    'agent_id': launch_result.agent_id if 'launch_result' in dir() else None,
    'agent_arn': launch_result.agent_arn if 'launch_result' in dir() else None,
    'region': REGION,
    'deployment_time': datetime.now().isoformat(),
    'execution_role': ROLE_ARN
}

%store deployment_info
%store runtime
%store baseline_metrics
print("Deployment information saved for Module 4")

## Summary

In this module, you:

1. **Created execution role** with appropriate permissions for DynamoDB, Bedrock, and CloudWatch
2. **Deployed to AgentCore Runtime** using the Starter Toolkit
3. **Tested the production deployment** with sample queries
4. **Verified observability** with CloudWatch integration
5. **Tested session management** for multi-turn conversations

### Production Architecture

```
+---------------------------------------------------------+
|                    AgentCore Runtime                     |
|  +---------------------------------------------------+  |
|  |              Multi-Agent Application               |  |
|  |  +-----------+ +-----------+ +-----------+        |  |
|  |  |  Order    | |  Product  | |  Account  |        |  |
|  |  |  Agent    | |  Agent    | |  Agent    |        |  |
|  |  | (Haiku45) | | (Haiku45) | | (Haiku45) |        |  |
|  |  +-----+-----+ +-----+-----+ +-----+-----+        |  |
|  |        +-------------+-----------+                 |  |
|  |              +-------v-------+                     |  |
|  |              | Orchestrator  |                     |  |
|  |              | (Sonnet 4.5)  |                     |  |
|  |              +---------------+                     |  |
|  +---------------------------------------------------+  |
|                          |                              |
|  +-----------------------------------------------+     |
|  |              AgentCore Observability           |     |
|  |  CloudWatch Logs | X-Ray Traces | Metrics      |     |
|  +-----------------------------------------------+     |
+---------------------------------------------------------+
```

### Next Steps

In **Module 4**, we'll configure online evaluation, simulate drift scenarios, and implement the improvement loop.